In [4]:
### 模块二：`obsidian_sync_agent.py`
# **功能描述**：
# 这是作业二的核心实现。它是一个常驻后台的智能代理，整合了 `watchdog`（监听）、`shutil`/`pathlib`（文件操作）、`subprocess`（剪贴板）以及 `argparse`（配置参数）。它负责将 AI 生成的卡片同步到 Obsidian，并记录操作日志。

import argparse
import json
import logging
import shutil
import subprocess
import time
import sys
from pathlib import Path
from datetime import datetime
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
from jinja2 import Environment, FileSystemLoader
import time
import errno



# --- 配置与日志 ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("sync_agent.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)

#02定义大类ObsidianSyncAgent
class ObsidianSyncAgent(FileSystemEventHandler):
    def __init__(self, config):
        self.config = config
        self.obsidian_vault = Path(config['vault_path'])
        self.inbox_path = Path(config['inbox_path'])
        # 确保 Jinja2 模板加载器安全
        template_loader_path = config.get('template_dir', 'templates')
        if not Path(template_loader_path).exists():
            logging.warning(f"⚠️ 模板目录不存在: {template_loader_path}，将使用内存模板作为回退")
            # 这里可以定义一个默认的内建模板以防万一
        self.template_env = Environment(loader=FileSystemLoader(template_loader_path, encoding='utf-8'))
      
        # 确保目录存在
        self.obsidian_vault.mkdir(parents=True, exist_ok=True)
        self.inbox_path.mkdir(parents=True, exist_ok=True)

# 02-1 监听文件创建事件"""
    def on_created(self, event):

        if event.is_directory: return
      
        path = Path(event.src_path)
         # 优化：增加文件存在性检查，防止竞态条件
        if path.exists() and path.suffix == '.json' and path.parent == self.inbox_path:
            logging.info(f"📥 捕获新指令: {path.name}")
            # 异步处理，防止阻塞监听线程
            self.process_instruction(path)
            
            
            
#02-2 处理 AI 生成的 JSON 指令-对接inbox            定时的
    def process_instruction(self, json_path: Path):
        """处理 AI 生成的 JSON 指令"""
        for attempt in range(max_retries):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                break # 成功读取则跳出循环
            except (IOError, OSError) as e:
                if e.errno == errno.EACCES and attempt < max_retries - 1:
                    logging.warning(f"文件被占用，{0.5}秒后重试... ({attempt+1}/{max_retries})")
                    time.sleep(0.5)
                else:
                    logging.error(f"❌ 无法读取指令文件: {e}")
                    return
            except json.JSONDecodeError as e:
                logging.error(f"❌ JSON 格式错误: {e}")
                return
            except Exception as e:
                logging.error(f"❌ 处理指令时发生未知错误: {e}")
                return

        command = data.get('command')
        payload = data.get('payload', {})

        try:
            if command == 'write_card':
                self.execute_write_card(payload)
            elif command == 'clipboard_paste':
                self.execute_clipboard_paste(payload)
            elif command == 'organize_files':
                self.execute_organize(payload)
            else:
                logging.warning(f"❓ 未知指令: {command}")
                return
          
            # 处理完成后归档指令
            self.archive_file(json_path)

        except Exception as e:
            logging.error(f"❌ 执行指令失败: {e}")

    # --- 功能模块 ---

    #jinja01 
    def execute_write_card(self, payload):
        """
        作业二核心：将 AI 内容卡片写入 Obsidian
        支持 jinja2 渲染和 pathlib 操作
        """
        target_file = self.obsidian_vault / payload['target_file']
        content_data = payload['content']
        mode = payload.get('mode', 'append') # append/prepend
      
        # 1. 使用 Jinja2 渲染内容
        try:
            template = self.template_env.get_template('standard_card.j2')
            rendered_content = template.render(**content_data)
        except Exception as e:
            logging.error(f"❌ 模板渲染失败: {e}")
            return
      
        # 2. 确保父目录存在;
        target_file.parent.mkdir(parents=True, exist_ok=True)
        # 3. 写入文件 (增加了文件锁防御)
        write_mode = 'a' if mode == 'append' else 'w'
        try:
            with open(target_file, write_mode, encoding='utf-8', newline='\n') as f:
                if mode == 'append' and target_file.exists() and target_file.stat().st_size > 0:
                    f.write("\n\n---\n\n") # 增加分隔符
                f.write(rendered_content)
            logging.info(f"✅ 写入成功: {target_file}")
        except (IOError, PermissionError) as e:
            logging.error(f"❌ 文件写入失败 (可能被其他程序占用): {e}")
            
            
#jinja02: 利用 subprocess 调用系统剪贴板
    def execute_clipboard_paste(self, payload):
        """
        作业二扩展：
        """
        text_to_copy = payload.get('text', '')
      
        # 跨平台剪贴板支持
        try:
            if sys.platform == 'win32':
                # Windows 使用 clip 命令
                subprocess.run(['clip'], input=text_to_copy.strip().encode('utf-16'), check=True)
            elif sys.platform == 'darwin':
                # macOS 使用 pbcopy
                subprocess.run(['pbcopy'], input=text_to_copy.strip().encode('utf-8'), check=True)
            else:
                # Linux (xclip)
                subprocess.run(['xclip', '-selection', 'clipboard'], input=text_to_copy.strip().encode('utf-8'), check=True)
          
            logging.info("📋 内容已复制到系统剪贴板")
        except Exception as e:
            logging.error(f"剪贴板操作失败: {e}")
            
            
#jinja03      初步实现了弱化版index
    def execute_organize(self, payload):
        """
        作业二扩展：文件整理与双向链接维护
        """
        src = Path(payload['src'])
        dst = self.obsidian_vault / payload['dst']
      

        if src.exists():
            try:            # 这里可以添加逻辑，扫描其他 md 文件更新 [[链接]]
            # 简化示例：仅记录日志
                shutil.move(str(src), str(dst))
                logging.info(f"📂 文件已移动: {src} -> {dst}")
                logging.info("🔗 建议运行双向链接检查脚本更新引用")
            except Exception as e:
                logging.error(f"❌ 文件移动失败: {e}")
        else:
            logging.warning(f"⚠️ 源文件不存在: {src}")

            
    def archive_file(self, file_path):
        """将处理完的指令移入 processed 文件夹"""
        processed_dir = self.inbox_path / 'processed'
        processed_dir.mkdir(exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        new_name = f"{timestamp}_{file_path.name}"
        
        try:
            # 先尝试重命名，如果跨设备则使用 copy + unlink
            if file_path.parent != processed_dir:
                shutil.copy2(file_path, processed_dir / new_name)
                file_path.unlink()
            else:
                file_path.rename(processed_dir / new_name)
        except Exception as e:
            logging.error(f"❌ 归档文件失败: {e}")


def main():
    # --- 参数解析 ---
    parser = argparse.ArgumentParser(description="Obsidian Knowledge Sync Agent")
    parser.add_argument('--config', type=str, default='config.json', help='配置文件路径')
    parser.add_argument('--vault', type=str, help='手动指定 Obsidian 库路径 (覆盖配置文件)')
    parser.add_argument('--scan-interval', type=int, default=1, help='监听扫描间隔(秒)')
  
    args = parser.parse_args()

    # 加载配置               据说json模式为{command: "write_card", payload:{...}}
    try:
        with open(args.config, 'r', encoding='utf-8') as f:
            config = json.load(f)
    except FileNotFoundError:
        # 默认配置兜底
        config = {
            "vault_path": "E:/Briah",
            "inbox_path": "E:/Briah/inbox",
            "template_dir": "E:/Briah/templates"
        }
        logging.warning("⚠️ 未找到配置文件，使用默认配置")

    # 覆盖参数
    if args.vault:
        config['vault_path'] = args.vault

    # --- 启动监听 ---
    try:
        event_handler = ObsidianSyncAgent(config)
        observer = Observer()
        observer.schedule(event_handler, config['inbox_path'], recursive=False)
        observer.start()
      
        logging.info(f"🚀 Obsidian Sync Agent 已启动，正在监听: {config['inbox_path']}")
        logging.info(f"⚙️  当前配置: {config}")

        try:
            while True:
                time.sleep(args.scan_interval)
        except KeyboardInterrupt:
            observer.stop()
            logging.info("👋 正在停止监听...")
        observer.join()
        logging.info("👋 Agent 已退出")
        
    except Exception as e:
        logging.critical(f"❌ Agent 启动失败: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()



ModuleNotFoundError: No module named 'watchdog'

In [ ]:
### 使用说明

#### 1. 准备环境
# 在 `E:\Briah\scripts\` 目录下安装依赖：
# ```bash
# pip install watchdog jinja2
# ```

# #### 2. 创建配置文件 (`config.json`)
# 放在脚本同目录或通过参数指定：
# ```json
{
  "vault_path": "E:/Briah",
  "inbox_path": "E:/Briah/inbox",
  "template_dir": "E:/Briah/templates"
}

#### 3. 创建模板 (`templates/standard_card.j2`)
# jinja2
### 【{{ code }}】{{ title }}
# - **AI索引标签**：{% for tag in tags %}`{{ tag }}` {% endfor %}
# - **生成时间**：{{ timestamp }}

{{ body }}


#### 4. 运行脚本
# ```bash
# python obsidian_sync_agent.py --vault "E:/Briah"


#### 5. 触发作业（模拟 AI 写入）
# 手动在 `E:\Briah\inbox` 下创建一个 `test.json`：
# ```json
{
  "command": "write_card",
  "payload": {
    "target_file": "记忆库.md",
    "content": {
      "code": "999-TEST",
      "title": "自动化测试卡片",
      "tags": ["自动化", "Python"],
      "timestamp": "2023-10-27",
      "body": "这是通过 Watchdog 和 Jinja2 自动生成的卡片内容。"
    },
    "mode": "append"
  }
}

# **结果**：脚本会立即检测到文件，渲染模板，将内容追加到 `E:\Briah\记忆库.md`，并将 `test.json` 移入 `processed` 文件夹。

In [2]:
# --- 新增串联逻辑：写入完成后自动刷新索引 ---
try:
    # 直接调用 Index 模块的刷新函数（需确保 index_manager.py 在路径下）
    from index_manager import refresh_index # 假设 index 代码保存为此文件名
    refresh_result = refresh_index()
    print(f"🔄 索引已同步更新: {refresh_result}")
except ImportError as e:
    print(f"⚠️ 无法自动更新索引，请手动运行 Index 模块: {e}")
# --- 结束新增 ---

当前Python解释器路径: C:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
Python版本: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
已安装包列表: ['C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\Pythonwin']


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys     #补正
print("当前Python解释器路径:", sys.executable)
print("Python版本:", sys.version)
print("已安装包列表:", [p for p in sys.path if 'site-packages' in p])
# 在Jupyter Notebook单元格中执行
!pip install watchdog

当前Python解释器路径: C:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
Python版本: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
已安装包列表: ['C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\Pythonwin']



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import sys
print("当前Python解释器路径:", sys.executable)
print("Python版本:", sys.version)
print("已安装包路径:", [p for p in sys.path if 'site-packages' in p])
# 检查watchdog是否已安装
try:
    import watchdog
    print(f"watchdog已安装，版本: {watchdog.__version__}")
except ImportError:
    print("watchdog未安装到当前Python环境")

# 查看当前环境中的所有包
!pip list | findstr watchdog

当前Python解释器路径: C:\Users\Administrator\AppData\Local\Programs\Python\Python311\python.exe
Python版本: 3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]
已安装包路径: ['C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\win32\\lib', 'C:\\Users\\Administrator\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\Pythonwin']
watchdog未安装到当前Python环境
watchdog           6.0.0
